<a href="https://colab.research.google.com/github/zty2020china/IOS13-SimulateTouch/blob/master/%E7%84%A6%E7%82%89%E7%BB%BC%E5%90%88%E4%BB%BF%E7%9C%9F%E5%B9%B3%E5%8F%B0V1_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# -*- coding: utf-8 -*-
"""
焦炉专业封装软件 V1.6 - 饱和温湿联动集成版
功能特性：
1. 燃料温度与饱和含湿量(g/Nm3)深度绑定，支持自动带出水的质量。
2. 继承 V1.5 所有数据库穿透、CmHn细化及产能测算逻辑。
3. 适配 Mac 网络环境与 Gradio 6.0 最新语法。
"""

import os
import math
import gradio as gr
import pandas as pd
from pydantic import BaseModel

# Mac 环境网络补丁
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"

# ==========================================
# 领域数据模型与物理常数
# ==========================================
WATER_VAPOR_DENSITY = 804.0  # 标况下水蒸气密度 g/Nm3

# 用户提供的实验数据点 (温度: 体积比 m3/Nm3)
USER_MOISTURE_DATA = {
    20: 0.0235, 25: 0.0323, 30: 0.0436,
    35: 0.0587, 40: 0.0785, 45: 0.1043
}

class FuelGas(BaseModel):
    h2: float; ch4: float; co: float; cmhn: float; cmhn_c2h4_ratio: float
    h2s: float; co2: float; n2: float; o2: float; humidity_g_nm3: float; temperature_c: float

class AirCondition(BaseModel):
    excess_ratio: float; temperature_c: float; relative_humidity: float

class CoalSample(BaseModel):
    moisture_mt: float; coke_temp_c: float; raw_gas_temp_c: float

class OvenSchedule(BaseModel):
    oven_count: int; wet_coal_per_oven: float; turnaround_time_h: float; coking_rate: float

# ==========================================
# 模块 A：炉型数据库 (保持 V1.5 逻辑)
# ==========================================
class OvenDatabase:
    def __init__(self, csv_path="焦炉常用参数 - 工作表1.csv"):
        self.db = {}
        self.load_from_csv(csv_path)

    def load_from_csv(self, csv_path):
        if not os.path.exists(csv_path): return
        for enc in ['gbk', 'utf-8-sig', 'gb2312']:
            try:
                df = pd.read_csv(csv_path, encoding=enc, index_col=0)
                df_t = df.T
                for model_name, row_data in df_t.iterrows():
                    self.db[str(model_name).strip()] = {k.strip(): str(v) for k, v in row_data.items() if pd.notna(v)}
                break
            except: continue

    def get_models(self):
        return list(self.db.keys()) if self.db else ["内置标准炉型"]

    def get_params(self, name):
        return self.db.get(name, {})

oven_db = OvenDatabase()

# ==========================================
# 模块 B：计算引擎 (增强温湿联动)
# ==========================================
class CokeOvenEngine:
    @staticmethod
    def get_saturated_moisture_g_nm3(temp_c):
        """
        根据用户实测数据点进行物理量转化与插值拟合
        公式：Mass(g) = Vol(m3) * 804
        """
        # 使用指数拟合：Vol = 0.0084 * exp(0.0558 * temp)
        # 这是基于用户 20-45度数据拟合的最优物理曲线
        vol_m3 = 0.0084 * math.exp(0.0558 * temp_c)
        return vol_m3 * WATER_VAPOR_DENSITY

    @staticmethod
    def run_simulation(fuel: FuelGas, air: AirCondition, coal: CoalSample, schedule: OvenSchedule):
        # 1. 组分拆解与热值
        c2h4 = fuel.cmhn * fuel.cmhn_c2h4_ratio
        c6h6 = fuel.cmhn * (1.0 - fuel.cmhn_c2h4_ratio)
        lhv = (126.36 * fuel.co + 107.98 * fuel.h2 + 358.18 * fuel.ch4 +
               590.3 * c2h4 + 1400.0 * c6h6 + 234.0 * fuel.h2s)

        # 2. 燃烧需氧与空气
        o2_demand = (0.5 * fuel.h2 + 0.5 * fuel.co + 2.0 * fuel.ch4 +
                     3.0 * c2h4 + 7.5 * c6h6 + 1.5 * fuel.h2s - fuel.o2) / 100.0
        v0_dry_air = o2_demand / 0.21

        # 3. 湿空气修正 (基于环境条件)
        p_sat_air = 6.112 * math.exp((17.62 * air.temperature_c) / (243.12 + air.temperature_c))
        x_w_air = (air.relative_humidity / 100.0) * p_sat_air / 1013.25
        v_actual_wet_air = (air.excess_ratio * v0_dry_air) / (1 - x_w_air)

        # 4. 废气全组分追踪 (Nm3/Nm3燃料)
        v_h2o_combust = (fuel.h2 + 2*fuel.ch4 + 2*c2h4 + 3*c6h6 + fuel.h2s)/100.0
        v_h2o_fuel_phys = fuel.humidity_g_nm3 / WATER_VAPOR_DENSITY # 燃料物理水体积
        v_h2o_air = v_actual_wet_air * x_w_air

        v_act_flue = (fuel.co + fuel.co2 + fuel.ch4 + 2*c2h4 + 6*c6h6)/100.0 + \
                     (fuel.h2s/100.0) + (fuel.n2/100.0 + 0.79*air.excess_ratio*v0_dry_air) + \
                     (0.21*(air.excess_ratio-1)*v0_dry_air) + \
                     (v_h2o_combust + v_h2o_fuel_phys + v_h2o_air)

        # 5. 产能与热平衡 (8000h基准)
        dry_coal_per_oven = schedule.wet_coal_per_oven * (1 - coal.moisture_mt/100.0)
        annual_coke = (schedule.oven_count / schedule.turnaround_time_h) * 8000 * schedule.wet_coal_per_oven * (schedule.coking_rate/100.0)

        q2_water = (coal.moisture_mt/(100-coal.moisture_mt)) * (2490 + 1.98*coal.raw_gas_temp_c)
        q1_eff = (schedule.coking_rate/100.0) * (1.05 + 0.0003*coal.coke_temp_c) * coal.coke_temp_c
        q_dry_heat = (q1_eff + q2_water) / 0.60

        # 6. 单室废气量挂钩
        hourly_dry_kg = (schedule.oven_count / schedule.turnaround_time_h) * dry_coal_per_oven * 1000.0
        total_wg = (hourly_dry_kg * q_dry_heat / lhv) * v_act_flue

        return {
            "capacity": annual_coke,
            "metrics": {
                "【燃料】当前温度下饱和水质量": f"{fuel.humidity_g_nm3:.2f} g/Nm³",
                "【燃烧】低位发热量 LHV": f"{lhv:.0f} kJ/Nm³",
                "【燃烧】实际湿废气生成率": f"{v_act_flue:.3f} Nm³/Nm³",
                "【热工】综合干煤炼焦耗热量": f"{q_dry_heat:.0f} kJ/kg(干煤)",
                "【分配】单室设计废气排烟量": f"**{total_wg / schedule.oven_count:.0f} Nm³/h**"
            }
        }

# ==========================================
# 模块 C：Gradio UI & 交互逻辑
# ==========================================
def update_moisture_by_temp(temp, auto_lock):
    """当温度改变且开启自动锁定时，自动更新湿度"""
    if auto_lock:
        val = CokeOvenEngine.get_saturated_moisture_g_nm3(temp)
        return round(val, 2)
    return gr.update()

def load_oven_params(name):
    d = oven_db.get_params(name)
    return float(d.get("每孔炭化室装煤量（t）", 30)), float(d.get("周转时间", 20)), float(d.get("成焦率", 75))

def run_calc_ui(fuel_tp, h2, ch4, co, cmhn, c2h4_r, h2s, co2, n2, o2, g_hum, g_temp, alpha, air_t, air_rh, c_mt, c_temp, r_temp, o_cnt, w_coal, t_time, c_rate):
    fuel = FuelGas(h2=h2, ch4=ch4, co=co, cmhn=cmhn, cmhn_c2h4_ratio=c2h4_r, h2s=h2s, co2=co2, n2=n2, o2=o2, humidity_g_nm3=g_hum, temperature_c=g_temp)
    air = AirCondition(excess_ratio=alpha, temperature_c=air_t, relative_humidity=air_rh)
    coal = CoalSample(moisture_mt=c_mt, coke_temp_c=c_temp, raw_gas_temp_c=r_temp)
    schedule = OvenSchedule(oven_count=o_cnt, wet_coal_per_oven=w_coal, turnaround_time_h=t_time, coking_rate=c_rate)

    res = CokeOvenEngine.run_simulation(fuel, air, coal, schedule)
    report = f"### 🎯 项目年产能估算：<span style='color:red'>{res['capacity']/10000:.2f} 万吨/年</span>\n\n"
    report += "| 计算维度 | 结果数值 |\n| :--- | :--- |\n"
    for k, v in res["metrics"].items(): report += f"| {k} | {v} |\n"
    return report

with gr.Blocks(title="焦炉设计大师 V1.6") as app:
    gr.Markdown("# 🏭 焦炉热工仿真平台 V1.6 (饱和温湿联动版)")

    with gr.Row():
        with gr.Column(scale=4):
            with gr.Accordion("📦 炉型与生产规模", open=True):
                model_dd = gr.Dropdown(choices=oven_db.get_models(), label="型号加载", value=oven_db.get_models()[0])
                with gr.Row():
                    o_cnt = gr.Number(value=65, label="孔数"); w_coal = gr.Number(label="装煤量(t)"); t_time = gr.Number(label="周转时间(h)"); c_rate = gr.Number(label="成焦率(%)")

            with gr.Accordion("💧 燃料温湿特性 (实测数据联动)", open=True):
                with gr.Row():
                    g_temp = gr.Slider(20, 50, value=35, label="燃料煤气温度 (℃)")
                    auto_lock = gr.Checkbox(value=True, label="自动锁定饱和湿度 (基于实测点)")
                g_hum = gr.Number(value=47.2, label="燃料含湿量 (g/Nm³)")
                gr.Markdown("<small>注：开启自动后，输入温度将自动匹配水的质量（1Nm³干煤气对应水重）。</small>")

            with gr.Accordion("🔥 燃烧组分与环境", open=False):
                with gr.Row(): h2 = gr.Number(58.0, label="H2"); ch4 = gr.Number(25.0, label="CH4"); co = gr.Number(6.0, label="CO")
                with gr.Row(): cmhn = gr.Number(2.5, label="CmHn"); c2h4_r = gr.Slider(0,1,0.8, label="C2H4占比"); h2s = gr.Number(0.5, label="H2S")
                with gr.Row(): co2 = gr.Number(2.5, label="CO2"); n2 = gr.Number(5.0, label="N2"); o2 = gr.Number(0.5, label="O2")
                with gr.Row(): alpha = gr.Slider(1.0, 1.5, 1.2, label="过剩系数"); air_t = gr.Number(25, label="气温"); air_rh = gr.Slider(0,100,60, label="空气RH%")

            with gr.Accordion("🍂 煤质与调火", open=False):
                c_mt = gr.Slider(5, 15, 10, label="煤水分%"); c_temp = gr.Number(1050, label="推焦温"); r_temp = gr.Number(750, label="荒煤气温")

            btn = gr.Button("🚀 开启全流程机理仿真", variant="primary")

        with gr.Column(scale=5):
            res_md = gr.Markdown("### 📑 待算工程报告\n点击左侧按钮生成...")

    # 联动逻辑
    model_dd.change(load_oven_params, [model_dd], [w_coal, t_time, c_rate])
    g_temp.change(update_moisture_by_temp, [g_temp, auto_lock], [g_hum])
    auto_lock.change(update_moisture_by_temp, [g_temp, auto_lock], [g_hum])

    btn.click(run_calc_ui,
             [gr.Text(visible=False), h2, ch4, co, cmhn, c2h4_r, h2s, co2, n2, o2, g_hum, g_temp, alpha, air_t, air_rh, c_mt, c_temp, r_temp, o_cnt, w_coal, t_time, c_rate],
             [res_md])

if __name__ == "__main__":
    app.launch(server_name="127.0.0.1", server_port=7860)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3f2fdcb8c9b5feee23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
